# Data Cleaning

The objective of this notebook is to clean up the data and export it in `parquet` to the `data/processed` folder.

What need to be cleaned/changed:
- Objects with sentinel values (`-9999`) will be removed with `pd.dropna`. There are only 3 objects with sentinel values, therefore it is okay to remove.
- Convert `class` type from `str` to `category`.
- Remove useless collumns.
- Add the diference between adjacent photometric variables as new columns (`u_g`, `g_r`, `r_i` and `i_z`).

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import sys

from pyprojroot import here

sys.path.append(str(here()))

from src.exist.config import DATA_PATH, output_data_dir

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
quality = pd.DataFrame(
    {
        "dtype": df.dtypes,
        "unique_values": df.nunique(),
        "null_count": df.isna().sum(),
        "sentinel_-9999_count": (df == -9999).sum(),
    }
)

In [ ]:
quality

In [ ]:
dropped_collumns = [
    "obj_ID",
    "alpha",
    "delta",
    "run_ID",
    "rerun_ID",
    "cam_col",
    "field_ID",
    "spec_obj_ID",
    "plate",
    "MJD",
    "fiber_ID",
]

df = df.drop(dropped_collumns, axis=1)

In [ ]:
df[["u", "g", "z"]] = df[["u", "g", "z"]].replace(-9999, np.nan)

In [ ]:
df = df.dropna()

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df["class"] = df["class"].astype("category")

In [ ]:
df.info()

In [ ]:
df["u_g"] = df["u"] - df["g"]
df["g_r"] = df["g"] - df["r"]
df["r_i"] = df["r"] - df["i"]
df["i_z"] = df["i"] - df["z"]

In [ ]:
assert (df[["u", "g", "z"]] == -9999).sum().sum() == 0

In [ ]:
assert df.isna().sum().sum() == 0

In [ ]:
df.to_parquet(output_data_dir, engine="pyarrow", index=False)